# 04 — Model Training (v3)

Улучшения для повышения AUC:
- **Новые признаки**: multi-horizon momentum (3d/10d/20d), gap return, 52w high/low distance, vol-of-vol, RSI slope
- **Threshold target**: `returns[+1] > 0.3%` вместо `> 0` (убираем шумные ~нулевые дни)
- **Cross-sectional ранги** (7 признаков)
- Итого: ~43 признака на модель

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'backend'))

PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR    = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TICKERS        = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META', 'JPM', 'JNJ', 'V']
TRAIN_RATIO    = 0.8
RETURN_THRESH  = 0.003

print(f'Target: next-day return > {RETURN_THRESH:.1%}')

## 1. Загрузка данных

In [ ]:
def load_ticker(ticker: str) -> pd.DataFrame:
    path = PROCESSED_DIR / ticker / 'technical.parquet'
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_parquet(path)
    df['date']   = pd.to_datetime(df['date'])
    df['ticker'] = ticker
    return df.sort_values('date').reset_index(drop=True)


def make_target(df: pd.DataFrame, threshold: float = 0.003) -> pd.DataFrame:
    df = df.copy()
    next_ret = df['returns'].shift(-1)
    df['target'] = (next_ret > threshold).astype(int)
    return df.dropna(subset=['target', 'returns'])


frames = []
for ticker in TICKERS:
    df = load_ticker(ticker)
    if df.empty:
        print(f'[SKIP] {ticker}: нет данных')
        continue
    df = make_target(df, RETURN_THRESH)
    frames.append(df)
    print(f'{ticker}: {len(df)} строк  target={df["target"].mean():.3f}')

if not frames:
    raise RuntimeError('Нет данных! Запустите 02_feature_engineering.ipynb')

combined = pd.concat(frames, ignore_index=True).sort_values('date').reset_index(drop=True)
print(f'\nВсего: {len(combined)} строк  Positive rate: {combined["target"].mean():.3f}')

## 2. Cross-sectional ранги

In [ ]:
rank_source = ['returns', 'returns_3d', 'returns_10d', 'rsi', 'momentum',
               'volatility', 'volume_ratio', 'macd_hist', 'bb_pct', 'dist_52w_high']

for feat in rank_source:
    if feat in combined.columns:
        combined[f'{feat}_rank'] = combined.groupby('date')[feat].rank(pct=True)

CROSS_FEATURES = [f'{f}_rank' for f in rank_source if f in combined.columns]
print(f'Cross-sectional ({len(CROSS_FEATURES)}): {CROSS_FEATURES}')

## 3. Train / Test split

In [ ]:
split_idx = int(len(combined) * TRAIN_RATIO)
train = combined.iloc[:split_idx]
test  = combined.iloc[split_idx:]

print(f'Train: {len(train)}  ({train["date"].min().date()} – {train["date"].max().date()})  pos={train["target"].mean():.3f}')
print(f'Test:  {len(test)}   ({test["date"].min().date()} – {test["date"].max().date()})   pos={test["target"].mean():.3f}')

## 4. Обучение

In [ ]:
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    roc_auc_score, brier_score_loss, average_precision_score,
)

from app.models.baseline_lr import BaselineLRModel
from app.models.model_a import ModelA
from app.models.model_b import ModelB
from app.models.model_c import ModelC
from app.models.model_d import ModelD

models = {
    'baseline_lr': BaselineLRModel(),
    'model_a':     ModelA(),
    'model_b':     ModelB(),
    'model_c':     ModelC(),
    'model_d':     ModelD(),
}

NON_FEATURE = {'date', 'ticker', 'target', 'close'}


def train_model(model, X_tr, y_tr, avail):
    inner = model.model
    if hasattr(inner, 'scale_pos_weight'):
        pos = int(y_tr.sum())
        neg = int((y_tr == 0).sum())
        inner.set_params(scale_pos_weight=neg / pos if pos > 0 else 1.0)
    inner.fit(X_tr[avail], y_tr)
    model.is_trained = True
    model._nb_features = avail


def proba(model, X, avail):
    return model.model.predict_proba(X[avail])[:, 1]


def evaluate(model, X_te, y_te, avail):
    p    = proba(model, X_te, avail)
    pred = (p >= 0.5).astype(int)
    return {
        'f1':        round(f1_score(y_te, pred, zero_division=0), 4),
        'roc_auc':   round(roc_auc_score(y_te, p), 4),
        'precision': round(precision_score(y_te, pred, zero_division=0), 4),
        'recall':    round(recall_score(y_te, pred, zero_division=0), 4),
        'brier':     round(brier_score_loss(y_te, p), 4),
        'pr_auc':    round(average_precision_score(y_te, p), 4),
    }


results   = {}
all_avail = {}

for name, model in models.items():
    avail = [
        f for f in model.feature_set + CROSS_FEATURES
        if f in combined.columns and f not in NON_FEATURE
    ]
    all_avail[name] = avail

    X_tr = train[avail].fillna(0)
    X_te = test[avail].fillna(0)
    y_tr = train['target']
    y_te = test['target']

    print(f'[{name}] ({len(avail)} признаков)...', end=' ')
    train_model(model, X_tr, y_tr, avail)

    m = evaluate(model, X_te, y_te, avail)
    results[name] = m
    model.save(MODELS_DIR / f'{name}.joblib')
    print(f'F1={m["f1"]:.3f}  AUC={m["roc_auc"]:.3f}  Brier={m["brier"]:.4f}')

print('\nГотово!')

## 5. Метрики и графики

In [ ]:
metrics_df = pd.DataFrame(results).T
print(metrics_df[['f1', 'roc_auc', 'precision', 'recall', 'brier', 'pr_auc']].round(4).to_string())

colors = ['#6366f1', '#22c55e', '#f59e0b', '#ef4444', '#06b6d4']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

metrics_df['f1'].plot(kind='bar', ax=axes[0], color=colors[:len(metrics_df)], width=0.6)
axes[0].set_title('F1 Score')
axes[0].set_ylim(0, 1)
axes[0].axhline(0.5, color='white', linestyle='--', alpha=0.4, label='random')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=30)

metrics_df['roc_auc'].plot(kind='bar', ax=axes[1], color=colors[:len(metrics_df)], width=0.6)
axes[1].set_title('ROC-AUC')
axes[1].set_ylim(0.4, 1)
axes[1].axhline(0.5, color='white', linestyle='--', alpha=0.4, label='random')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'Сравнение моделей (threshold={RETURN_THRESH:.1%})', fontsize=13)
plt.tight_layout()
plt.show()

## 6. ROC-кривые

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))

for (name, model), color in zip(models.items(), colors):
    if name not in results:
        continue
    avail = all_avail[name]
    p     = proba(model, test[avail].fillna(0), avail)
    fpr, tpr, _ = roc_curve(test['target'], p)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={results[name]["roc_auc"]:.3f})')

ax.plot([0, 1], [0, 1], 'w--', alpha=0.3, label='random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC-кривые')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature importance

In [ ]:
if 'model_a' in results:
    model_a = models['model_a']
    avail   = all_avail['model_a']
    fi = pd.Series(model_a.model.feature_importances_, index=avail).sort_values(ascending=False)

    fi_cross = fi[[f for f in fi.index if f.endswith('_rank')]]
    fi_base  = fi[[f for f in fi.index if not f.endswith('_rank')]]

    print(f'Доля cross-sectional: {fi_cross.sum() / fi.sum():.1%}')
    print(f'Топ-5: {fi.head().index.tolist()}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    fi.head(20).plot(kind='barh', ax=axes[0], color='#6366f1')
    axes[0].set_title('Model A — топ-20 признаков')
    axes[0].invert_yaxis()

    combined_fi = pd.concat([fi_cross, fi_base.head(10)]).sort_values()
    bar_colors  = ['#22c55e' if f.endswith('_rank') else '#6366f1' for f in combined_fi.index]
    combined_fi.plot(kind='barh', ax=axes[1], color=bar_colors)
    axes[1].set_title('Cross-sectional (зелёный) vs технические (синий)')
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

## 8. Calibration curves

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))

for (name, model), color in zip(models.items(), colors):
    if name not in results:
        continue
    avail = all_avail[name]
    p     = proba(model, test[avail].fillna(0), avail)
    prob_true, prob_pred = calibration_curve(test['target'], p, n_bins=10)
    ax.plot(prob_pred, prob_true, marker='o', color=color, label=name, lw=1.5)

ax.plot([0, 1], [0, 1], 'w--', alpha=0.4, label='идеальная')
ax.set_xlabel('Predicted probability')
ax.set_ylabel('True probability')
ax.set_title('Calibration curves')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()